In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

if not os.path.exists('razor-pay-assurance'):
    !git clone https://github.com/emilmariagiby/razor-pay-assurance.git

sys.path.append(os.path.abspath('razor-pay-assurance/backend'))
from app.learning.adapters.ieee_cis_adapter import IeeeCisAdapter


In [ ]:
ieee_adapter = IeeeCisAdapter()
ieee_path = None

# Search Kaggle input directory dynamically so we never guess the path wrong
if os.path.exists('/kaggle/input'):
    for dirname, _, filenames in os.walk('/kaggle/input'):
        for filename in filenames:
            if filename == 'train_transaction.csv':
                ieee_path = os.path.join(dirname, filename)
                break
elif os.path.exists('../backend/data/external/ieee_cis/train_transaction.csv'):
    ieee_path = '../backend/data/external/ieee_cis/train_transaction.csv'

if not ieee_path:
    raise FileNotFoundError("\n\nCRITICAL: Could not find 'train_transaction.csv'!\nYou MUST click 'Add Data' on the right panel in Kaggle and add the 'IEEE-CIS Fraud Detection' dataset to this session.")

print(f"Loading IEEE-CIS data from: {ieee_path}")
all_records = ieee_adapter.ingest(ieee_path)
print(f"Total canonical ExternalBehavioralRecords loaded: {len(all_records)}")


In [ ]:
X = []
y = []
feature_names = []

if all_records:
    feature_names = sorted(all_records[0].behavioral_features.keys())
    for record in all_records:
        X.append(record.to_vector())
        y.append(record.external_fraud_label)

X = np.array(X)
y = np.array(y)
print(f"Feature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training set: {X_train.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")


In [ ]:
print("Training Random Forest model...")
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
print("Training complete.")


In [ ]:
y_pred = rf_model.predict(X_val)
y_prob = rf_model.predict_proba(X_val)[:, 1]

print("Classification Report:")
print(classification_report(y_val, y_pred, zero_division=0))

roc_auc = roc_auc_score(y_val, y_prob)
print(f"ROC-AUC Score: {roc_auc:.4f}")


In [ ]:
MODEL_PATH = 'external_behavior_model.joblib'
joblib.dump(rf_model, MODEL_PATH)
print(f"Model successfully exported to: {MODEL_PATH}")
print("You can now download this file from Kaggle's output directory and place it in your local 'backend/data/models/' folder!")
